# 05 — Collection Operations

Notebook 04 gave you the data shapes. Notebook 03 gave you function values. This notebook puts them together: the **higher-order operations** that take a collection and a function and return a new collection.

Almost every Scala collection — `List`, `Vector`, `Set`, `Map`, even `Array` — supports the same operation vocabulary: `map`, `filter`, `flatMap`, `fold`, `groupBy`, `zip`, and friends. Learn the vocabulary once and you can apply it everywhere. Spark's `Dataset` and `DataFrame` APIs are the same operations, lifted to a distributed setting.

## `map` — transform each element

`map` applies a function to every element of a collection and returns a new collection of the same shape, filled with the results. The function's type determines the result element type.

In [ ]:
val nums = List(1, 2, 3, 4)

nums.map(x => x * x)        // List(1, 4, 9, 16)
nums.map(_ * 2)             // List(2, 4, 6, 8)
nums.map(_.toString)        // List[String] = List("1", "2", "3", "4")

Three things to register about `map`:

- The output collection has the **same length** as the input.
- The output **element type** comes from the function's return type. Mapping `Int => String` turns `List[Int]` into `List[String]`.
- The output **collection type** stays the same — `List.map` returns a `List`, `Vector.map` returns a `Vector`, `Set.map` returns a `Set`.

## `filter` — keep what matches

`filter` takes a predicate (a function returning `Boolean`) and returns a new collection containing only the elements where the predicate is true. `filterNot` is the inverse.

In [ ]:
val nums = List(1, 2, 3, 4, 5, 6)

nums.filter(_ % 2 == 0)     // List(2, 4, 6)
nums.filterNot(_ % 2 == 0)  // List(1, 3, 5)
nums.exists(_ > 4)          // true
nums.forall(_ > 0)          // true
nums.count(_ % 2 == 0)      // 3

Four close cousins of `filter` worth knowing:

- `exists(p)` — is there at least one element where `p` is true?
- `forall(p)` — is `p` true for every element?
- `count(p)` — how many elements satisfy `p`?
- `find(p)` — return the first matching element wrapped in `Option`. `None` if nothing matches.

`map` and `filter` together cover the vast majority of everyday transformations. The next operation lets you do things they can't.

## `flatMap` — transform and flatten

`flatMap` is `map` followed by `flatten`. The function you give it returns a *collection* per input element, and `flatMap` concatenates those collections into one.

In [ ]:
val words = List("hi there", "how are you")

words.map(_.split(" ").toList)
// List(List("hi", "there"), List("how", "are", "you"))

words.flatMap(_.split(" ").toList)
// List("hi", "there", "how", "are", "you")

`flatMap` shows up whenever you have a *one-to-many* transformation: each input element produces zero or more output elements, and you want a single flat collection at the end. This same shape generalises to `Option` and `Either` in notebook 09 — where `flatMap` is how you chain operations that might fail.

## `fold` and `reduce` — collapse to a single value

`map`, `filter`, and `flatMap` produce new collections. `fold` and `reduce` produce a single value by combining elements pairwise.

`reduce` takes a binary combining function and applies it across the collection. Works only on non-empty collections.

In [ ]:
val nums = List(1, 2, 3, 4)

nums.reduce(_ + _)        // 10  — like ((1 + 2) + 3) + 4
nums.reduce(_ max _)      // 4
// List.empty[Int].reduce(_ + _)   // throws — reduce needs at least one element

`foldLeft` takes an explicit **starting value** (the *zero*) and a combining function. It works on empty collections — it just returns the zero.

In [ ]:
nums.foldLeft(0)(_ + _)              // 10
nums.foldLeft(1)(_ * _)              // 24  — factorial-ish
nums.foldLeft("")((acc, n) => acc + n.toString)   // "1234"

List.empty[Int].foldLeft(0)(_ + _)   // 0  — safe on empty input

Reading the signature `foldLeft(zero)(op)`:

- `zero` is what you start with when the collection is empty.
- `op` is `(accumulator, element) => newAccumulator`. It receives the running result so far and the next element, and returns the next running result.
- The two parameter lists are deliberate — it's the multiple-parameter-list trick from notebook 03. It lets the compiler infer the accumulator's type from `zero`, so you rarely have to write it.

For common folds, Scala ships shortcuts: `sum`, `product`, `min`, `max`. Reach for those first. `foldLeft` is for the cases the shortcuts don't cover.

In [ ]:
nums.sum         // 10
nums.product     // 24
nums.min         // 1
nums.max         // 4
nums.mkString(", ")    // "1, 2, 3, 4"

## Grouping — `groupBy`, `partition`

`groupBy` takes a function `A => K` and returns a `Map[K, List[A]]` — elements bucketed by what the function returns.

In [ ]:
val words = List("ant", "bat", "ape", "bee", "cat")

words.groupBy(_.head)
// Map('a' -> List("ant", "ape"),
//     'b' -> List("bat", "bee"),
//     'c' -> List("cat"))

`partition` is the boolean version: split a collection into the elements that match a predicate and the elements that don't.

In [ ]:
val nums = List(1, 2, 3, 4, 5, 6)
val (evens, odds) = nums.partition(_ % 2 == 0)
// evens: List(2, 4, 6)
// odds:  List(1, 3, 5)

Note the tuple destructuring on the left side of the `val` — partition returns a `(List[A], List[A])`, and we unpack both at once. This is the tuple syntax from notebook 04 in action.

`groupBy` is what Spark's `groupBy` is built on, lifted to a distributed setting. The local in-memory version is exactly the same shape: pick a key function, get a Map of key to elements.

## Combining collections — `zip`, `zipWithIndex`

`zip` pairs elements of two collections position by position, stopping at the shorter one.

In [ ]:
val names = List("alice", "bob", "cara")
val ages = List(30, 25, 41)

names.zip(ages)
// List(("alice", 30), ("bob", 25), ("cara", 41))

names.zipWithIndex
// List(("alice", 0), ("bob", 1), ("cara", 2))

`zipWithIndex` is the every-day trick: pair each element with its index so you can use both inside a `map` or `filter`.

## Sorting and ordering

Three flavours, depending on what's natural at the call site:

In [ ]:
val xs = List(3, 1, 4, 1, 5, 9, 2, 6)

xs.sorted                     // List(1, 1, 2, 3, 4, 5, 6, 9) — uses natural ordering
xs.sorted(using Ordering.Int.reverse)  // descending — Scala 3 explicit given
xs.sortBy(x => -x)            // sort by a derived key
xs.sortWith(_ > _)            // sort with a comparison function

- `sorted` uses the natural ordering when there is one (`Int`, `String`, etc.). For your own types, see notebook 11 (givens).
- `sortBy(f)` is the everyday choice: define a *key function* and sort by what it returns.
- `sortWith` takes an explicit comparator; reach for it when the comparison is more complex than a single key.

## `for ... yield` — sugar, not a feature

Here is the unifier. A `for ... yield` expression is **pure syntactic sugar** that desugars to a chain of `map`, `flatMap`, and `withFilter`. Once you know that, the syntax stops feeling like magic.

The simplest case — one generator, no filter — desugars to `map`:

In [ ]:
val nums = List(1, 2, 3, 4)

// for-yield form
val squares = for n <- nums yield n * n

// desugars to exactly:
val same = nums.map(n => n * n)

// both: List(1, 4, 9, 16)

Add an `if` guard, and it desugars to `withFilter` followed by `map`:

In [ ]:
val evenSquares =
  for
    n <- nums
    if n % 2 == 0
  yield n * n
// List(4, 16)

// desugars to:
val same = nums.withFilter(_ % 2 == 0).map(n => n * n)

Add a **second generator**, and it desugars to `flatMap` plus an inner `map`:

In [ ]:
val pairs =
  for
    x <- List(1, 2, 3)
    y <- List("a", "b")
  yield (x, y)
// List((1,"a"), (1,"b"), (2,"a"), (2,"b"), (3,"a"), (3,"b"))

// desugars to:
val same =
  List(1, 2, 3).flatMap(x =>
    List("a", "b").map(y => (x, y)))

Read the desugaring rule once and you will see it everywhere:

```
  last generator + yield   ->  map
  inner generator          ->  flatMap (the outer one)
  if guard                 ->  withFilter
  no yield (just `do`)     ->  foreach
```

That is the entire `for` comprehension specification. Everything else is shape and indentation. The same syntax works for any type that defines `map`, `flatMap`, and `withFilter` — including `Option`, `Either`, `Future`, and Spark's `Dataset`. Notebook 09 will use `for` over `Option` chains; notebook 13 will use it over `Future`.

## Views — laziness for chains

Every operation you've seen so far is **strict** — it produces a new collection right away. Chain three `map`s and you allocate three intermediate collections. For long pipelines on big inputs, that's wasteful.

A **view** is a lazy projection of a collection. Operations on a view are not executed until you ask for the result. Call `.view` to enter laziness, `.toList` (or `.toVector`, etc.) to force evaluation.

In [ ]:
val nums = (1 to 1_000_000).toList

// strict — allocates two intermediate Lists of 1 million elements
val a = nums.map(_ * 2).filter(_ > 100).take(3)

// lazy — no intermediate collections; stops as soon as 3 hits are found
val b = nums.view.map(_ * 2).filter(_ > 100).take(3).toList

// both: List(102, 104, 106)

When to use views:

- Long pipelines where intermediate collections would be large.
- Early-exit operations like `take(n)`, `find`, `exists`, where you don't need the full input.

When *not* to use them: short pipelines on small data. The lazy machinery itself has overhead; for a 10-element list with two transformations, strict evaluation is faster *and* simpler to read.

Note: Spark's RDD and `Dataset` are *always* lazy. Every transformation is a recipe; nothing runs until an *action* like `collect` or `count`. You will feel the same pattern that views give you locally, just at cluster scale.

## Putting it together

A small pipeline that uses several of the operations you just met. Goal: from a list of orders, compute total revenue per customer.

In [ ]:
case class Order(customer: String, item: String, amount: Double)
// (case classes formally arrive in notebook 07 — using one here for readability)

val orders = List(
  Order("alice", "book",   12.0),
  Order("bob",   "pen",     2.5),
  Order("alice", "coffee",  4.5),
  Order("cara",  "book",   12.0),
  Order("bob",   "book",   12.0),
)

val totalsByCustomer: Map[String, Double] =
  orders
    .groupBy(_.customer)
    .view
    .mapValues(_.map(_.amount).sum)
    .toMap

// Map("alice" -> 16.5, "bob" -> 14.5, "cara" -> 12.0)

Walk through the pipeline:

1. `groupBy(_.customer)` buckets orders into `Map[String, List[Order]]`.
2. `.view` enters laziness so the next `mapValues` doesn't allocate an intermediate `Map`.
3. `mapValues(_.map(_.amount).sum)` replaces each list of orders with the sum of its amounts.
4. `.toMap` forces the view back into a strict `Map`.

Compare this to a hand-written `var`-and-loop version: half the lines, no mutation, and the logic reads top to bottom in the order it conceptually happens. That readability is the whole point of the vocabulary.

## What's next

Two notebooks finished the data half of Scala: shapes (04) and operations (05). The next two notebooks pivot to **modeling**. Notebook 06 introduces classes, objects, and traits — how Scala packages behaviour into reusable types. Notebook 07 then introduces case classes and enums, which are the form most domain modelling in Scala uses.